<a href="https://colab.research.google.com/github/AnushkaPachauri/Opencv--projects/blob/main/Shape_Detection_and_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import cv2 as cv
from google.colab.patches import cv2_imshow
import numpy as np

In [ ]:
# INPUT
img_path = input("Enter image path: ")
img = cv.imread(img_path)

# Convert to grayscale
gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
#cv2_imshow(gray)

In [ ]:
# Removing noise (decrypting image)
denoise = cv.medianBlur(gray, 5)
#cv2_imshow(denoise)

In [ ]:
kernel = np.ones((3,3),np.uint8)
clean = cv.morphologyEx(denoise, cv.MORPH_OPEN, kernel)
#cv2_imshow(clean)

In [ ]:
_, thresh = cv.threshold(clean, 1, 255, cv.THRESH_BINARY)
#cv2_imshow(thresh)
kernel = np.ones((2,2),np.uint8)
dilate = cv.dilate(thresh,kernel,iterations=2)
#cv2_imshow(dilate)

In [ ]:
kernel = np.ones((8,8),np.uint8)
close = cv.morphologyEx(dilate, cv.MORPH_CLOSE, kernel)
#cv2_imshow(close)


In [ ]:
contours,_ = cv.findContours(close,cv.RETR_EXTERNAL,cv.CHAIN_APPROX_SIMPLE)

In [ ]:
shapes = []
detected_objects = []
rect_count = 0
circle_count = 0
output_img = cv.cvtColor(close, cv.COLOR_GRAY2BGR)

In [ ]:
for cnt in contours:
    area = cv.contourArea(cnt)

    # Ignoring the small noise
    if area < 500:
        continue

    peri = cv.arcLength(cnt, True)
    approx = cv.approxPolyDP(cnt, 0.03 * peri, True)
    num_corners = len(approx)

    if num_corners == 4:
       shape = "Rectangle"
       rect_count += 1

    else:
        shape = "Circle"
        circle_count += 1

    detected_objects.append((cnt, shape))

if rect_count > circle_count:
    rect_color = (0, 255, 0)   # Green (BGR)
    circle_color = (0, 0, 255) # Red (BGR)
else:
    rect_color = (0, 0, 255)   # Red
    circle_color = (0, 255, 0) # Green


for cnt, shape in detected_objects:

    if shape == "Rectangle":
        color = rect_color
    elif shape == "Circle":
        color = circle_color

    shapes.append(shape)
    cv.drawContours(output_img, [cnt], -1, color, 10)

print("shapes : ",shapes)
cv2_imshow(output_img)